## Train a linear probe on Raven unit embeddings
---
Embed exported Raven units with Perch2, optionally cluster background clips, and train a linear classifier using folder names as class labels.

Author: Danelle Cline dcline@mbari.org

### Set paths
Choose the config YAML, dataset directory, and output directory that contains (or will contain) exported Raven unit clips.

In [1]:
from pathlib import Path
config_yaml_path =  "../config.yaml"
output_path = Path("output_hb")
dataset_path = Path("dataset_hb")
output_path.mkdir(parents=True, exist_ok=True)

### Load configuration
Load and verify the config. Later cells use it for Perch2 window settings and model output paths.

In [2]:
from perchtopic.config import Config
try:
    config = Config(config_yaml_path, output_path)
    config.verify()
except Exception as e:
    print(e)


CONFIG
wav_path: ../config.yaml 
output_path: output_hb/doc
model_path: output_hb/model
perch_hop_seconds: 0.5
perch_audio_seconds: 5.0
perch_window_fill: fill
PERCH_TIME_BIN_SECONDS: 5


### Embed Raven units
Run the Perch2 ONNX model over cached unit clips and collect embeddings for classification.

In [3]:
# Compute embeddings from the Perch2 ONNX model
from pathlib import Path
from perchtopic.cache import UnitCacheKey
from perchtopic.config import Config
from perchtopic.embed import Embedding
from perchtopic.classify import build_model

config = Config(dataset_path, output_path)
emb = Embedding(UnitCacheKey.from_config(config), Path("perch_v2.onnx"))
embeddings = emb.run()

Loading ONNX model perch_v2.onnx
Loading 5545 clips; keep 5s then fill to 160000 samples
  [1/5545] MARS_20161221_000046_SongSession_32kHz_HPF5Hz.selections_?.139438837.139454732.sel.1433.ch01.wav
  [2/5545] MARS_20161221_000046_SongSession_32kHz_HPF5Hz.selections_?.280488062.280517596.sel.2918.ch01.wav
  [3/5545] MARS_20161221_000046_SongSession_32kHz_HPF5Hz.selections_?.281600208.281648142.sel.2927.ch01.wav
  [4/5545] MARS_20161221_000046_SongSession_32kHz_HPF5Hz.selections_?.283451688.283478318.sel.2946.ch01.wav
  [5/5545] MARS_20161221_000046_SongSession_32kHz_HPF5Hz.selections_?.283839995.283864204.sel.2947.ch01.wav
  [6/5545] MARS_20161221_000046_SongSession_32kHz_HPF5Hz.selections_?.283899548.283923273.sel.2948.ch01.wav
  [7/5545] MARS_20161221_000046_SongSession_32kHz_HPF5Hz.selections_?.283949418.283971206.sel.2949.ch01.wav
  [8/5545] MARS_20161221_000046_SongSession_32kHz_HPF5Hz.selections_?.283983310.284002193.sel.2950.ch01.wav
  [9/5545] MARS_20161221_000046_SongSession_32k

### Cluster background clips
Density-cluster background unit embeddings so similar noise/background sounds share a cluster label. `assign_noise=True` assigns every file a cluster number.

In [4]:
from perchtopic.cluster.density import cluster_directory

c = cluster_directory(
    output_path / "units" / "MARS_20161221_000046_SongSession_32kHz_HPF5Hz"/ "background",
    # output_path / "units" / "sequence"/ "background",
    model_path="perch_v2.onnx",
    config=config,          # optional; uses perch window settings
    assign_noise=True,      # default: every file gets a cluster number
)
print(f"Found {c.n_clusters} background cluster(s)")

/Users/dcline/Dropbox/code/soundscape/perchtopic/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Embedding 25 clips in output_hb/units/MARS_20161221_000046_SongSession_32kHz_HPF5Hz/background
UMAP (25, 1536) -> 15-D (cosine)
HDBSCAN clustering
Found 1 clusters (assigned 8 noise to nearest cluster)
Moved 25 clips to output_hb/units/MARS_20161221_000046_SongSession_32kHz_HPF5Hz/background/background_0
Found 1 background cluster(s)


### Train and sanity-check the linear probe
Build a linear model from unique unit-folder labels, train on the embeddings, save the checkpoint, and print training accuracy. This in-sample check is a sanity test, not a held-out evaluation.

In [6]:
import torch
from perchtopic.train_utils import labels_to_one_hot

clips = sorted((config.output_path / "units").rglob("*.wav"))
labels = [clip.parent.name for clip in clips]
unique_labels = list(set(labels))
one_hot_labels, class_to_index = labels_to_one_hot(labels)

model = build_model(len(unique_labels))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X = torch.from_numpy(embeddings).to(device)
y = one_hot_labels.to(device)
n = len(X)

epochs = 1000
batch_size = 64

for epoch in range(epochs):
    perm = torch.randperm(n)
    epoch_loss = 0.0
    for lo in range(0, n, batch_size):
        idx = perm[lo : lo + batch_size]
        epoch_loss += model.train_step(X[idx], y[idx]) * len(idx)
    epoch_loss /= n
    if epoch % max(1, epochs // 10) == 0 or epoch == epochs - 1:
        print(f"epoch {epoch:4d}  loss {epoch_loss:.5f}")

model.save(config.model_path / "linear_model.pt", classes=sorted(set(labels)))

# Test the model against its own data; this is a contrived example for sanity check
# Best practice is to have a separate test set. This should return a 100.0% training accuracy
idx_to_class = {i: name for name, i in class_to_index.items()}
pred_idx = model.predict_proba(X).argmax(dim=1).cpu().numpy()
pred_labels = [idx_to_class[int(i)] for i in pred_idx]
n_correct = sum(p == t for p, t in zip(pred_labels, labels))
print(f"train accuracy: {n_correct}/{len(labels)} ({n_correct / len(labels):.1%})")
for clip, true, pred in zip(clips, labels, pred_labels):
    mark = "ok" if true == pred else "MISS"
    print(f"{mark}  {clip.name}  true={true}  pred={pred}")

print("Done")

epoch    0  loss 0.0747
epoch   20  loss 0.0097
epoch   40  loss 0.0059
epoch   60  loss 0.0040
epoch   80  loss 0.0029
epoch  100  loss 0.0022
epoch  120  loss 0.0016
epoch  140  loss 0.0013
epoch  160  loss 0.0010
epoch  180  loss 0.0009
epoch  200  loss 0.0007
epoch  220  loss 0.0006
epoch  240  loss 0.0005
epoch  260  loss 0.0004
epoch  280  loss 0.0003
epoch  300  loss 0.0003
epoch  320  loss 0.0002
epoch  340  loss 0.0002
epoch  360  loss 0.0002
epoch  380  loss 0.0001
epoch  400  loss 0.0001
epoch  420  loss 0.0001
epoch  440  loss 0.0001
epoch  460  loss 0.0001
epoch  480  loss 0.0001
epoch  500  loss 0.0001
epoch  520  loss 0.0001
epoch  540  loss 0.0000
epoch  560  loss 0.0000
epoch  580  loss 0.0000
epoch  600  loss 0.0000
epoch  620  loss 0.0000
epoch  640  loss 0.0000
epoch  660  loss 0.0000
epoch  680  loss 0.0000
epoch  700  loss 0.0000
epoch  720  loss 0.0000
epoch  740  loss 0.0000
epoch  760  loss 0.0000
epoch  780  loss 0.0000
epoch  800  loss 0.0000
epoch  820  loss